# Preparing the PDFs
We took all published policies from Odoo whose language is marked "English" and downloaded their metadata including data (PDF) links into an excel sheet.

In [ ]:
import pandas as pd
import os
import requests
import glob

cwd = os.getcwd()
data_dir = cwd+"/data"

odoo_df = pd.read_excel(data_dir+"/ASPECT_PolicieswIDs_3Mar26.xlsx")

We create a list of tuples containing the Odoo policy ID and data/PDF link

In [3]:
data_links = []
for idx in odoo_df.index:
    odoo_id = int(odoo_df.loc[idx,'ID'])
    data_links.append((odoo_id, odoo_df.loc[idx,"Data Link"]))
for link in data_links:
    print(link)

(246, 'https://www.unccd.int/sites/default/files/2022-02/cop21add1_SF_EN.pdf')
(247, 'https://www.cbd.int/doc/decisions/cop-15/cop-15-dec-04-en.pdf')
(248, 'https://unfccc.int/files/meetings/paris_nov_2015/application/pdf/paris_agreement_english_.pdf?gad_source=1&gclid=Cj0KCQjw4MSzBhC8ARIsAPFOuyVVJZxGg4CVx3hv5fYVikwpS8_3YKhP5o_Y_nRPXY1_n4h656QcKVsaAgOHEALw_wcB')
(249, 'https://docs.un.org/en/A/RES/70/1')
(250, 'https://wedocs.unep.org/bitstream/handle/20.500.11822/31813/ERDStrat.pdf?sequence=1&isAllowed=y')
(251, 'https://qna.files.parliament.uk/qna-attachments/1716677/original/Agricultural%20Transition%20Plan%20update%20January%202024%20-%20GOV.UK%20-%20HL4488.pdf')
(252, 'https://assets.publishing.service.gov.uk/media/5f6b6da6e90e076c182d508d/023_15482_Environment_agency_digitalAW_Strategy.pdf')
(253, 'https://www.ramsar.org/sites/default/files/documents/library/current_convention_text_e.pdf')
(254, 'https://documents.un.org/doc/undoc/gen/k19/011/03/pdf/k1901103.pdf')
(256, 'https://

We download the PDFs from the links, using the IDs as the PDF names.

In [ ]:
success = []
failure = []
for triple in data_links:
    if triple[0]> 590:
        try:
            response = requests.get(triple[1])
            pdf = open(f"{data_dir}/pdfs/{triple[0]}.pdf", 'wb')
            pdf.write(response.content)
            pdf.close()
            success.append(triple)
            print(triple[0], "succeeded")
        except:
            print("File", triple[0], "failed")
            failure.append(triple)
print(f"\nSuccesses: {len(success)}; Failures: {len(failure)}")
print("Failure list:")
for fail in failure:
    print(f"{fail[0]}")
''''''
#failures: [590, 592, 873, 950, 951, 983, 1050, 1055, 1172, 1173, 1175, 1176, 1177]

Some PDFs failed to be collected and others appeared to succeed but in fact did not save correctly

In [ ]:
# from "failures" in downloading PDFs
failed_to_download = [590, 592, 873, 950, 951, 983, 1050, 1055, 1172, 1173, 1175, 1176, 1177]

# failures determined from the PDF sizes
failed_to_access = []

downloaded_pdfs = glob.glob(f"{data_dir}/pdfs/*")
size_tuples = []
for pdf_addr in downloaded_pdfs:
    size_tuples.append((int(pdf_addr.split("/")[-1].split(".")[0]), os.path.getsize(pdf_addr)))
# will need to determine which were likely corrupted
sorted(size_tuples, key=lambda x: x[1])

[(1019, 0),
 (1021, 0),
 (875, 0),
 (1034, 0),
 (1024, 0),
 (1023, 0),
 (1033, 0),
 (1026, 0),
 (1027, 0),
 (261, 0),
 (257, 0),
 (1018, 0),
 (256, 0),
 (1028, 0),
 (264, 0),
 (263, 0),
 (258, 0),
 (1076, 212),
 (925, 212),
 (248, 212),
 (1096, 212),
 (866, 212),
 (944, 212),
 (978, 212),
 (1142, 565),
 (895, 839),
 (249, 3505),
 (1195, 4545),
 (250, 4545),
 (286, 4546),
 (303, 6795),
 (961, 7260),
 (610, 7272),
 (251, 7350),
 (344, 10595),
 (1079, 12912),
 (1072, 12912),
 (1082, 12912),
 (1045, 12912),
 (1074, 12912),
 (1064, 12912),
 (1061, 20082),
 (253, 22771),
 (853, 39316),
 (275, 52087),
 (364, 91973),
 (812, 103077),
 (350, 136665),
 (315, 140067),
 (254, 152739),
 (901, 174067),
 (817, 212382),
 (932, 228910),
 (913, 229702),
 (917, 311782),
 (912, 317623),
 (247, 329142),
 (933, 352522),
 (246, 371371),
 (1205, 399873),
 (958, 497206),
 (1147, 513990),
 (960, 558400),
 (1014, 655266),
 (945, 669128),
 (603, 773503),
 (335, 958427),
 (1179, 990373),
 (1022, 1168941),
 (810, 11

Let's prefilter policies containing certain keywords for our first annotation runthrough

In [13]:
key_docs = {"peatland":[], "climate":[], "cap":[]}
for idx in odoo_df.index:
    for kwd in list(key_docs):
        if kwd in str(odoo_df.loc[idx, "Name"]).lower():
            key_docs[kwd].append(int(odoo_df.loc[idx, "ID"]))
        elif kwd in str(odoo_df.loc[idx, "Name (Native Language)"]).lower():
            key_docs[kwd].append(int(odoo_df.loc[idx, "ID"]))
key_docs

{'peatland': [254, 316, 363, 1071, 1182],
 'climate': [294,
  298,
  299,
  300,
  315,
  320,
  335,
  336,
  343,
  344,
  345,
  347,
  349,
  354,
  357,
  364,
  590,
  599,
  601,
  602,
  608,
  812,
  814,
  816,
  861,
  862,
  865,
  895,
  929,
  949,
  952,
  977,
  994,
  1009,
  1014,
  1020,
  1022,
  1061,
  1076,
  1094,
  1169],
 'cap': [319]}